Import

In [1]:
import os
import random
import shutil
from collections import defaultdict

random.seed(42)

RAW_DATASET = "dataset"
OUTPUT_DATASET = "model_dataset"

SPLIT = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}


Helper

In [2]:
def mkdir(path):
    os.makedirs(path, exist_ok=True)


Parse Raw Folder Names

In [3]:
def parse_folder_name(folder):
    """
    Examples:
    ka            -> base=ka, pili=none
    kaal_pilla   -> base=ka, pili=al_pilla
    kakombuwa    -> base=ka, pili=kombuwa
    """
    known_pili = ["al_pilla", "kombuwa", "diga_pilla", "kodiya"]

    for p in known_pili:
        if folder.endswith(p):
            base = folder.replace(p, "")
            return base, p

    return folder, "none"


Scan Raw Dataset

In [4]:
dataset = defaultdict(list)

for folder in os.listdir(RAW_DATASET):
    folder_path = os.path.join(RAW_DATASET, folder)
    if not os.path.isdir(folder_path):
        continue

    base, pili = parse_folder_name(folder)

    images = [
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    for img in images:
        dataset[(base, pili)].append(os.path.join(folder_path, img))

print("Detected classes:")
for k in dataset:
    print(k, "->", len(dataset[k]))


Detected classes:
('අ', 'none') -> 12
('ආ', 'none') -> 1
('ඇ', 'none') -> 5
('ඈ', 'none') -> 1
('ඉ', 'none') -> 1
('ඊ', 'none') -> 1
('උ', 'none') -> 1
('ඌ', 'none') -> 1
('එ', 'none') -> 1
('ඒ', 'none') -> 1
('ඔ', 'none') -> 1
('ඕ', 'none') -> 1
('ක', 'none') -> 38
('ක්', 'none') -> 1
('කා', 'none') -> 1
('කැ', 'none') -> 1
('කෑ', 'none') -> 1
('කි', 'none') -> 1
('කී', 'none') -> 1
('කු', 'none') -> 1
('කූ', 'none') -> 1
('කෙ', 'none') -> 1
('කේ', 'none') -> 1
('කො', 'none') -> 1
('කෝ', 'none') -> 1
('ග', 'none') -> 41
('ග්', 'none') -> 1
('ගා', 'none') -> 1
('ගැ', 'none') -> 5
('ගෑ', 'none') -> 1
('ගි', 'none') -> 1
('ගී', 'none') -> 1
('ගු', 'none') -> 1
('ගූ', 'none') -> 1
('ගෙ', 'none') -> 1
('ගේ', 'none') -> 1
('ගො', 'none') -> 1
('ගෝ', 'none') -> 1
('ඟ', 'none') -> 1
('ඟා', 'none') -> 1
('ඟි', 'none') -> 1
('ඟී', 'none') -> 1
('ඟු', 'none') -> 1
('ඟූ', 'none') -> 1
('ඟෙ', 'none') -> 1
('ඟේ', 'none') -> 1
('ඟො', 'none') -> 1
('ඟෝ', 'none') -> 1
('ච', 'none') -> 1
('ච්', 'none') 

Create Output Folder Structure

In [5]:
bases = sorted(set(b for b, _ in dataset))
pilis = sorted(set(p for _, p in dataset))

for split in SPLIT:
    for b in bases:
        for p in pilis:
            mkdir(os.path.join(OUTPUT_DATASET, split, b, p))


Split & Copy Images

In [6]:
def split_and_copy():
    for (base, pili), images in dataset.items():
        random.shuffle(images)
        n = len(images)

        n_train = int(n * SPLIT["train"])
        n_val = int(n * SPLIT["val"])

        splits = {
            "train": images[:n_train],
            "val": images[n_train:n_train+n_val],
            "test": images[n_train+n_val:]
        }

        for split, imgs in splits.items():
            for src in imgs:
                dst = os.path.join(
                    OUTPUT_DATASET, split, base, pili,
                    os.path.basename(src)
                )
                shutil.copy2(src, dst)

    print("✅ Dataset split completed!")

split_and_copy()


✅ Dataset split completed!


Sanity Check

In [7]:
def count_images(root):
    total = 0
    for b in os.listdir(root):
        for p in os.listdir(os.path.join(root, b)):
            total += len(os.listdir(os.path.join(root, b, p)))
    return total

for s in SPLIT:
    print(s, count_images(os.path.join(OUTPUT_DATASET, s)))


train 208
val 36
test 359


Label Maps (Auto)

In [8]:
base_map = {b:i for i,b in enumerate(bases)}
pili_map = {p:i for i,p in enumerate(pilis)}

print("Base map:", base_map)
print("Pili map:", pili_map)


Base map: {'අ': 0, 'ආ': 1, 'ඇ': 2, 'ඈ': 3, 'ඉ': 4, 'ඊ': 5, 'උ': 6, 'ඌ': 7, 'එ': 8, 'ඒ': 9, 'ඔ': 10, 'ඕ': 11, 'ක': 12, 'ක්': 13, 'කා': 14, 'කැ': 15, 'කෑ': 16, 'කි': 17, 'කී': 18, 'කු': 19, 'කූ': 20, 'කෙ': 21, 'කේ': 22, 'කො': 23, 'කෝ': 24, 'ග': 25, 'ග්': 26, 'ගා': 27, 'ගැ': 28, 'ගෑ': 29, 'ගි': 30, 'ගී': 31, 'ගු': 32, 'ගූ': 33, 'ගෙ': 34, 'ගේ': 35, 'ගො': 36, 'ගෝ': 37, 'ඟ': 38, 'ඟා': 39, 'ඟි': 40, 'ඟී': 41, 'ඟු': 42, 'ඟූ': 43, 'ඟෙ': 44, 'ඟේ': 45, 'ඟො': 46, 'ඟෝ': 47, 'ච': 48, 'ච්': 49, 'චා': 50, 'චැ': 51, 'චෑ': 52, 'චි': 53, 'චී': 54, 'චු': 55, 'චූ': 56, 'චෙ': 57, 'චේ': 58, 'චො': 59, 'චෝ': 60, 'ජ': 61, 'ජ්': 62, 'ජා': 63, 'ජැ': 64, 'ජෑ': 65, 'ජි': 66, 'ජී': 67, 'ජු': 68, 'ජූ': 69, 'ජෙ': 70, 'ජේ': 71, 'ජෝ': 72, 'ට': 73, 'ට්': 74, 'ටා': 75, 'ටැ': 76, 'ටෑ': 77, 'ටි': 78, 'ටී': 79, 'ටු': 80, 'ටූ': 81, 'ටෙ': 82, 'ටේ': 83, 'ටො': 84, 'ටෝ': 85, 'ඩ': 86, 'ඩ්': 87, 'ඩා': 88, 'ඩැ': 89, 'ඩෑ': 90, 'ඩි': 91, 'ඩී': 92, 'ඩු': 93, 'ඩූ': 94, 'ඩෙ': 95, 'ඩේ': 96, 'ඩො': 97, 'ඩෝ': 98, 'ණ': 99, 'ණ්': 100, 'ණා': 10

OpenCV Preprocessing (ConvNeXt)

In [9]:
import cv2
import numpy as np

def tight_crop(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    coords = cv2.findNonZero(th)
    x, y, w, h = cv2.boundingRect(coords)
    crop = img[y:y+h, x:x+w]

    size = max(w, h)
    padded = np.zeros((size, size, 3), dtype=np.uint8)
    padded[
        (size-h)//2:(size-h)//2+h,
        (size-w)//2:(size-w)//2+w
    ] = crop

    return padded


def preprocess(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (128, 128))
    img = np.stack([img, img, img], axis=-1)
    return img / 255.0


Load Dataset (Multi-Head)

In [10]:
def load_split(split):
    X, yL, yP = [], [], []

    root = os.path.join(OUTPUT_DATASET, split)

    for b in bases:
        for p in pilis:
            folder = os.path.join(root, b, p)

            if not os.path.exists(folder):
                continue

            for img_name in os.listdir(folder):

                if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                    continue

                img_path = os.path.join(folder, img_name)
                img = cv2.imread(img_path)

                if img is None:
                    print("⚠️ Skipping unreadable image:", img_path)
                    continue

                img = preprocess(tight_crop(img))

                X.append(img)
                yL.append(base_map[b])
                yP.append(pili_map[p])

    return np.array(X), np.array(yL), np.array(yP)


Check empty folders

In [11]:
for split in ["train", "val", "test"]:
    for b in bases:
        for p in pilis:
            path = os.path.join(OUTPUT_DATASET, split, b, p)
            if os.path.exists(path) and len(os.listdir(path)) == 0:
                print("Empty:", path)


Empty: model_dataset\train\ආ\none
Empty: model_dataset\train\ඈ\none
Empty: model_dataset\train\ඉ\none
Empty: model_dataset\train\ඊ\none
Empty: model_dataset\train\උ\none
Empty: model_dataset\train\ඌ\none
Empty: model_dataset\train\එ\none
Empty: model_dataset\train\ඒ\none
Empty: model_dataset\train\ඔ\none
Empty: model_dataset\train\ඕ\none
Empty: model_dataset\train\ක්\none
Empty: model_dataset\train\කා\none
Empty: model_dataset\train\කැ\none
Empty: model_dataset\train\කෑ\none
Empty: model_dataset\train\කි\none
Empty: model_dataset\train\කී\none
Empty: model_dataset\train\කු\none
Empty: model_dataset\train\කූ\none
Empty: model_dataset\train\කෙ\none
Empty: model_dataset\train\කේ\none
Empty: model_dataset\train\කො\none
Empty: model_dataset\train\කෝ\none
Empty: model_dataset\train\ග්\none
Empty: model_dataset\train\ගා\none
Empty: model_dataset\train\ගෑ\none
Empty: model_dataset\train\ගි\none
Empty: model_dataset\train\ගී\none
Empty: model_dataset\train\ගු\none
Empty: model_dataset\train\ගූ\

Class Weights (MANDATORY)

In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

NUM_LETTERS = len(base_map)
NUM_PILI = len(pili_map)

# Base letter class weights
letter_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(yL_train),
    y=yL_train
)

# Pili class weights
pili_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(yP_train),
    y=yP_train
)

letter_cw = dict(enumerate(letter_weights))
pili_cw = dict(enumerate(pili_weights))

print("Letter class weights:", letter_cw)
print("Pili class weights:", pili_cw)


NameError: name 'yL_train' is not defined